# Proyecto Isla Verde — Punto de Entrada Reproducible

### Particionamiento en Zonas de Falla de la Red Eléctrica del ICE mediante QAOA

**Equipo Isla Verde** — Quantathon 2026, Reto 1 · Open Quantum Institute (OQI)

---

***Resumen—*** *Este notebook es el punto de entrada reproducible de la fase
clásica de datos del proyecto. Prepara el entorno a partir de `requirements.txt`
y ejecuta la etapa de modelado (`modelador_red.py`), que construye el grafo de
la red de transmisión de 230 kV del Instituto Costarricense de Electricidad
(ICE), extrae las instancias NISQ (MVP-8 / STD-12 / LARGE-16), formula el
Max-Cut restringido como QUBO/Ising y calcula las líneas base clásicas. Los
artefactos resultantes (`scratch/isla_verde_*.json`) son la entrada de los tres
notebooks de análisis de la suite.*

***Términos—*** *reproducibilidad, QUBO, Max-Cut, red eléctrica, islanding.*

> **Nota de alcance.** El módulo `modelador_red.py` está en proceso de revisión.
> Por eso este punto de entrada llega **hasta la etapa del modelador** y se
> detiene ahí: las etapas cuánticas aguas abajo (QAOA y comparación) **no** se
> encadenan todavía, y se documentan por separado en los notebooks
> `01_qubo.ipynb`, `02_linea_base.ipynb` y `03_qaoa.ipynb`. Una vez estabilizado
> el modelador, este notebook será el orquestador único de todo el pipeline.


## I. Entorno

Se instalan las dependencias pineadas. En un entorno limpio esto garantiza que
las versiones de las librerías coincidan con las usadas para generar los
resultados reportados (requisito de reproducibilidad de la rúbrica).

In [ ]:
# --- Setup: permite correr este notebook desde notebooks/ o desde la raiz ---
import os, sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "qaoa.py").exists() and (ROOT.parent / "qaoa.py").exists():
    ROOT = ROOT.parent
    os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("Directorio de trabajo:", ROOT)

In [ ]:
!pip install -r requirements.txt

## II. Etapa Clásica: Modelado de la Red

Se ejecuta `modelador_red.py`, que realiza de forma determinista (semilla fija):

1. Ingesta y normalización de los CSV oficiales del ICE (`data/`).
2. Construcción del grafo físico de 230 kV y extracción de las instancias.
3. Formulación QUBO/Ising del Max-Cut restringido.
4. Cálculo de las líneas base clásicas (fuerza bruta, voraz, Goemans–Williamson).
5. Suite de autoverificación (`QUBO == fórmula directa`, `Ising == QUBO`, etc.).

> Si falta `data/` con los CSV, el script cae a un *proxy* sintético de 8 nodos y
> solo genera `mvp8`; `std12` y `large16` se omiten. Para el escalado completo se
> requieren los CSV del ICE en `data/`.

In [ ]:
!python modelador_red.py --data-dir data --out-dir scratch --seed 42

## III. Verificación de Artefactos

Se listan los archivos generados y se carga el índice para confirmar qué
instancias quedaron disponibles para las etapas de análisis.

In [ ]:
import json
from pathlib import Path

scratch = Path("scratch")
print("Artefactos en scratch/:")
for f in sorted(scratch.glob("isla_verde_*.json")):
    print("  ", f.name)

idx = json.loads((scratch / "isla_verde_index.json").read_bytes().decode("utf-8"))
print("\nInstancias disponibles:", list(idx["instances"].keys()))
print("Errores de instancia   :", idx.get("instance_errors", {}))
print("Fuente de datos        :", idx["metadata"]["source"])
print("Semilla                :", idx["metadata"]["seed"])

In [ ]:
import pandas as pd

filas = []
for tier, fname in idx["instances"].items():
    d = json.loads((scratch / fname).read_bytes().decode("utf-8"))
    bl = d["baselines"]["maxcut"]
    opt = bl["brute_force"]["cut"]
    filas.append({
        "instancia": tier,
        "n (qubits)": len(d["variable_order"]),
        "aristas": len(d["edges"]),
        "óptimo (fuerza bruta)": round(opt, 4),
        "greedy r": round(bl["greedy"]["cut"] / opt, 4),
        "GW r": round(bl["goemans_williamson"]["cut"] / opt, 4) if bl["goemans_williamson"] else None,
    })
resumen = pd.DataFrame(filas).set_index("instancia")
resumen

## IV. Frontera del Pipeline

Hasta aquí llega el punto de entrada reproducible en su estado actual. Las
etapas cuánticas se ejecutan de forma independiente:

| Notebook | Contenido |
|---|---|
| `01_qubo.ipynb` | Formulación y verificación QUBO/Ising del Max-Cut. |
| `02_linea_base.ipynb` | Líneas base clásicas: fuerza bruta, voraz y Goemans–Williamson. |
| `03_qaoa.ipynb` | Circuito QAOA, optimización local y validación en el emulador H2. |

> **Pendiente (tras la revisión del modelador):** encadenar aquí el barrido de
> QAOA y la comparación cuántico-vs-clásico para reproducir, con un solo
> comando, cada figura y cifra del informe.